In [ ]:
!pip install -q qiskit qiskit-aer
import qiskit, qiskit_aer
print(qiskit.__version__, qiskit_aer.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 82.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 3.9 MB/s eta 0:00:00
2.5.1 0.17.2


In [ ]:
from google.colab import files
import os
os.makedirs('qec', exist_ok=True); os.makedirs('data/snapshots', exist_ok=True); os.makedirs('runs', exist_ok=True)
open('qec/__init__.py','w').close()
up = files.upload()   # pick: tier0.py, layouts.py, tier1.py, snapshots.zip

Saving snapshots.zip to snapshots.zip


In [ ]:
import shutil, zipfile, glob
for f in ['tier0.py','layouts.py','tier1.py']:
    if f in up: shutil.move(f, f'qec/{f}')
zipfile.ZipFile('snapshots.zip').extractall('data/snapshots')
print(len(glob.glob('data/snapshots/*.json')), "snapshots")

21 snapshots


In [ ]:
!python -m qec.tier1 --snapshots "data/snapshots/*_converted.json" --quick --shots 1500

/usr/bin/python3: No module named qec.tier1


In [ ]:
import os
print("qec/:", os.listdir('qec'))
print("cwd:", [f for f in os.listdir('.') if f.endswith('.py')])

qec/: ['__pycache__', '__init__.py']
cwd: []


In [ ]:
import shutil
for f in ['tier0.py','layouts.py','tier1.py']:
    if os.path.exists(f): shutil.move(f, f'qec/{f}')
print(os.listdir('qec'))

['__pycache__', '__init__.py']


In [ ]:
from google.colab import files
import shutil, os
up = files.upload()
for f in list(up):
    if f.endswith('.py'):
        shutil.move(f, f'qec/{f}')
print(os.listdir('qec'))

Saving tier1.py to tier1.py
Saving tier0.py to tier0.py
Saving layouts.py to layouts.py
['tier0.py', 'tier1.py', 'layouts.py', '__pycache__', '__init__.py']


In [ ]:
!python -m qec.tier1 --snapshots "data/snapshots/*_converted.json" --quick --shots 1500

loaded 21 snapshots | rounds=3 shots=1500 holdout=3

21 unique cycles, 413 valid patches
held-out evaluation cycles: 3 (indices 18..20)

[1/3] held-out 2026-07-14T01:40:15  agree=False  (0s)
[2/3] held-out 2026-07-15T01:49:54  agree=False  (1s)
[3/3] held-out 2026-07-19T14:45:49  agree=False  (2s)


In [ ]:
import json, glob, sys
from qec import layouts, tier1
snaps=[json.load(open(f)) for f in sorted(glob.glob("data/snapshots/*_converted.json"))]
cy=layouts.unique_cycles(snaps)
adj=layouts.coupling_from_snapshot(cy[-1]); alive={int(q) for q in cy[-1]["qubits"]}
pats=layouts.enumerate_patches(adj,alive)
for T in (18,19,20):
    prior=cy[:T]; held=cy[T]
    inst={p:s for p in pats if (s:=layouts.instantaneous_score(prior[-1],p)) is not None}
    arch={p:s for p in pats if (s:=layouts.archive_score(prior,p)) is not None}
    pols={"today":layouts.rank(inst)[0][0],"arch":layouts.rank(arch)[0][0],"weak":layouts.rank(inst)[-1][0]}
    for name,patch in pols.items():
        print(f"T{T} {name} {patch} ...", flush=True)
        v=tier1.run_condition(held,patch,tier1.ENVELOPE[1],"ENC_ACTIVE",1,3,1500)
        print(f"   ok {v:.4f}", flush=True)
print("ALL SURVIVED")

T18 today (1, 2, 3, 16, 23) ...
   ok 0.0033
T18 arch (20, 21, 36, 41, 40) ...
   ok 0.0053
T18 weak (149, 150, 151, 152, 153) ...
   ok 0.0273
T19 today (1, 2, 3, 16, 23) ...
   ok 0.0020
T19 arch (20, 21, 36, 41, 40) ...
   ok 0.0107
T19 weak (146, 147, 148, 149, 150) ...
   ok 0.0120
T20 today (1, 2, 3, 16, 23) ...


In [ ]:
import json, glob
snaps=[json.load(open(f)) for f in sorted(glob.glob("data/snapshots/*_converted.json"))]
from qec import layouts
cy=layouts.unique_cycles(snaps)
for T in (19,20):
    s=cy[T]
    print("cycle",T,s.get("calibration_time"))
    for q in (1,2,3,16,23):
        p=s["qubits"][str(q)]
        print("  q%-3d T1=%8.2f T2=%8.2f ro=%.5f  T2>2T1=%s" %
              (q,p["T1_us"],p["T2_us"],p.get("readout_error",-1),p["T2_us"]>2*p["T1_us"]))
    cz=s["gates"]["cz"]
    for a,b in ((1,2),(2,3),(3,16),(16,23)):
        g=cz.get("(%d,%d)"%(a,b),cz.get("(%d,%d)"%(b,a)))
        print("  cz",(a,b),g)
    for gname in ("sx","x"):
        print("  %s errors:"%gname, {q:s["gates"].get(gname,{}).get(str(q)) for q in (1,2,3,16,23)})

cycle 19 None
  q1   T1=  235.25 T2=  232.08 ro=0.01465  T2>2T1=False
  q2   T1=  250.04 T2=  163.00 ro=0.00244  T2>2T1=False
  q3   T1=  168.30 T2=  130.44 ro=0.01831  T2>2T1=False
  q16  T1=  162.32 T2=  112.05 ro=0.01428  T2>2T1=False
  q23  T1=  224.17 T2=  187.55 ro=0.00537  T2>2T1=False
  cz (1, 2) 0.0025439386302968936
  cz (2, 3) 0.0019867974505203556
  cz (3, 16) 0.0023332140033108517
  cz (16, 23) 0.0033114655071456722
  sx errors: {1: 0.00025566049676903276, 2: 0.00021713138883134138, 3: 0.00021286651859677604, 16: 0.00046725629008084534, 23: 0.00021886989717162506}
  x errors: {1: 0.00025566049676903276, 2: 0.00021713138883134138, 3: 0.00021286651859677604, 16: 0.00046725629008084534, 23: 0.00021886989717162506}
cycle 20 None
  q1   T1=  149.05 T2=  195.38 ro=0.02600  T2>2T1=False
  q2   T1=  199.74 T2=  155.83 ro=0.00354  T2>2T1=False
  q3   T1=  181.08 T2=   62.53 ro=0.01477  T2>2T1=False
  q16  T1=  188.73 T2=  132.96 ro=0.01172  T2>2T1=False
  q23  T1=  132.69 T2=  129.

In [ ]:
from qec import layouts, tier1
import json, glob
snaps=[json.load(open(f)) for f in sorted(glob.glob("data/snapshots/*_converted.json"))]
cy=layouts.unique_cycles(snaps); held=cy[20]
for patch in [(1,2,3,16,23),(20,21,36,41,40)]:
    for st in (0,1):
        v=tier1.run_condition(held,patch,tier1.ENVELOPE[1],"ENC_PASSIVE",st,3,4000)
        print(patch, "|%d>"%st, "ENC_PASSIVE p_L=%.4f"%v, flush=True)
print("ENC_PASSIVE STABLE")

(1, 2, 3, 16, 23) |0> ENC_PASSIVE p_L=0.0045


In [ ]:
from qec import layouts, tier1, tier0
from qiskit_aer import AerSimulator
import json, glob
snaps=[json.load(open(f)) for f in sorted(glob.glob("data/snapshots/*_converted.json"))]
held=layouts.unique_cycles(snaps)[20]
patch=(1,2,3,16,23)

nm = tier1.build_patch_noise(held, patch, tier1.ENVELOPE[1])
sim = AerSimulator(noise_model=nm)          # built ONCE
qc0 = tier0.build_encoded(3,0,active=False)
qc1 = tier0.build_encoded(3,1,active=False)
for i in range(8):
    for st,qc in ((0,qc0),(1,qc1)):
        c = sim.run(qc, shots=1500).result().get_counts()
        print(i, st, "ok", len(c), flush=True)
print("REUSED SIMULATOR SURVIVED 16 RUNS")

0 0 ok 15


In [ ]:
from qec import layouts, tier1, tier0
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, thermal_relaxation_error, depolarizing_error, ReadoutError
import json, glob
snaps=[json.load(open(f)) for f in sorted(glob.glob("data/snapshots/*_converted.json"))]
held=layouts.unique_cycles(snaps)[20]
patch=(1,2,3,16,23)
qc1 = tier0.build_encoded(3,1,active=False)

def trial(label, nm):
    print(label, "...", flush=True)
    AerSimulator(noise_model=nm).run(qc1, shots=1500).result().get_counts()
    print("   OK", flush=True)

full = tier1.build_patch_noise(held, patch, tier1.ENVELOPE[1])
trial("full model on |1>", full)

full model on |1> ...
   OK


In [ ]:
from google.colab import files
import shutil, os
up = files.upload()          # pick qec_tier1_runner.py
for f in list(up):
    if f.endswith('.py'): shutil.move(f, 'qec/tier1_runner.py')
print(os.listdir('qec'))

Saving qec_tier1_runner.py to qec_tier1_runner.py
['tier1_runner.py', 'tier0.py', 'tier1.py', 'layouts.py', '__pycache__', '__init__.py']


In [ ]:
!python -m qec.tier1_runner --snapshots "data/snapshots/*_converted.json" --quick --shots 1500

21 snapshots | 3 held-out cycles | 9 cells | shots=1500
discordant cycles: 3/3

[1/9] T18|nominal|P_today|ENC_ACTIVE|1           p_L=0.0013 (1s)
[2/9] T18|nominal|P_archive|ENC_ACTIVE|1         p_L=0.0087 (2s)
[3/9] T18|nominal|P_weak|ENC_ACTIVE|1            p_L=0.0353 (3s)
[4/9] T19|nominal|P_today|ENC_ACTIVE|1           p_L=0.0027 (4s)
[5/9] T19|nominal|P_archive|ENC_ACTIVE|1         p_L=0.0093 (6s)
[6/9] T19|nominal|P_weak|ENC_ACTIVE|1            p_L=0.0073 (7s)
[7/9] T20|nominal|P_today|ENC_ACTIVE|1           CRASHED (8s)
[8/9] T20|nominal|P_archive|ENC_ACTIVE|1         p_L=0.0080 (9s)
[9/9] T20|nominal|P_weak|ENC_ACTIVE|1            p_L=0.0047 (10s)

TIER 1 (HELD-OUT, PROCESS-ISOLATED) GATE EVALUATION
cells: 8/9 ok, 1 failed (crash rate 11.1%)
  NOTE: failures are Aer native crashes; they are a logged
  Tier 1 deviation, not results. Affected keys in the JSON.

paired delta p_L on HELD-OUT cycles (archive - today;
negative favours the archive policy):
  nominal|state1             

In [ ]:
!python -m qec.tier1_runner --snapshots "data/snapshots/*_converted.json" --holdout 8 --shots 4000

21 snapshots | 8 held-out cycles | 432 cells | shots=4000
discordant cycles: 8/8

[1/432] T13|optimistic|P_today|BARE|0              p_L=0.0255 (1s)
[2/432] T13|optimistic|P_today|BARE|1              p_L=0.0490 (2s)
[3/432] T13|optimistic|P_today|ENC_PASSIVE|0       p_L=0.0025 (4s)
[4/432] T13|optimistic|P_today|ENC_PASSIVE|1       p_L=0.0047 (5s)
[5/432] T13|optimistic|P_today|ENC_ACTIVE|0        p_L=0.0020 (6s)
[6/432] T13|optimistic|P_today|ENC_ACTIVE|1        p_L=0.0018 (7s)
[7/432] T13|optimistic|P_archive|BARE|0            p_L=0.0057 (8s)
[8/432] T13|optimistic|P_archive|BARE|1            p_L=0.0450 (9s)
[9/432] T13|optimistic|P_archive|ENC_PASSIVE|0     p_L=0.0075 (11s)
[10/432] T13|optimistic|P_archive|ENC_PASSIVE|1     p_L=0.0092 (12s)
[11/432] T13|optimistic|P_archive|ENC_ACTIVE|0      p_L=0.0047 (13s)
[12/432] T13|optimistic|P_archive|ENC_ACTIVE|1      p_L=0.0063 (14s)
[13/432] T13|optimistic|P_weak|BARE|0               p_L=0.0105 (15s)
[14/432] T13|optimistic|P_weak|BARE|1 

In [ ]:
from google.colab import files
import shutil, os
up = files.upload()          # qec_diagnose_score.py
for f in list(up):
    if f.endswith('.py'): shutil.move(f, 'qec/diagnose_score.py')

Saving qec_diagnose_score.py to qec_diagnose_score.py


In [ ]:
!python -m qec.diagnose_score --snapshots "data/snapshots/*_converted.json"

131 usable ENC_ACTIVE cells from runs/tier1_partial.jsonl

--- ALL | state ALL  (n=131) ---
  readout_sum          +0.607  ############
  inv_T1_sum           +0.306  ######
  inv_T2_sum           +0.271  #####
  cz_err_sum           +0.344  ######
  hist_mean            +0.288  #####
  hist_variance        -0.114  ##
  hist_tail            -0.188  ###
  COMPOSITE_today      +0.282  #####
  COMPOSITE_archive    -0.114  ##

--- nominal | state 0  (n=21) ---
  readout_sum          +0.910  ##################
  inv_T1_sum           +0.390  #######
  inv_T2_sum           +0.372  #######
  cz_err_sum           +0.437  ########
  hist_mean            +0.274  #####
  hist_variance        -0.289  #####
  hist_tail            -0.292  #####
  COMPOSITE_today      +0.392  #######
  COMPOSITE_archive    -0.295  #####

--- nominal | state 1  (n=29) ---
  readout_sum          +0.910  ##################
  inv_T1_sum           +0.413  ########
  inv_T2_sum           +0.360  #######
  cz_err_sum        

In [ ]:
!python -m qec.layouts "data/snapshots/*_converted.json" --json-out runs/g1_A1.json

loaded 21 snapshot files

QEC-P1 GATE G1 — INTERVENTION DISTINCTNESS
unique calibration cycles : 21
valid 5-qubit patches     : 413
selection decisions       : 20

policy disagreement rate  : 100.0%
mean qubit overlap        : 0.65 / 5
expected discordant windows in 4: 4.0
mean score consequence when discordant: 1.22981

cycle                      agree  ovl   consequence
--------------------------------------------------------------------
2026-07-05T01:10:16-04:00  False  0/5       2.32800
2026-07-06T22:47:59-04:00  False  0/5       0.45243
2026-07-07T01:36:48-04:00  False  0/5       0.55801
2026-07-08T10:34:30-04:00  False  0/5       1.40171
2026-07-09T01:39:34-04:00  False  0/5       1.40696
2026-07-10T01:17:21-04:00  False  0/5       0.53478
2026-07-11T01:26:01-04:00  False  0/5       1.58566
2026-07-12T02:24:35-04:00  False  1/5       0.78622
2026-07-13T01:11:56-04:00  False  0/5       1.91464
2026-07-14T01:40:15-04:00  False  0/5       0.38855
2026-07-15T01:49:54-04:00  False  0/

In [ ]:
!rm -f runs/tier1_partial.jsonl
!python -m qec.tier1_runner --snapshots "data/snapshots/*_converted.json" --holdout 8 --shots 4000 --partial runs/tier1_A1_partial.jsonl --json-out runs/tier1_A1.json

21 snapshots | 8 held-out cycles | 432 cells | shots=4000
discordant cycles: 8/8

[1/432] T13|optimistic|P_today|BARE|0              p_L=0.0240 (1s)
[2/432] T13|optimistic|P_today|BARE|1              p_L=0.0465 (2s)
[3/432] T13|optimistic|P_today|ENC_PASSIVE|0       p_L=0.0040 (4s)
[4/432] T13|optimistic|P_today|ENC_PASSIVE|1       p_L=0.0030 (5s)
[5/432] T13|optimistic|P_today|ENC_ACTIVE|0        p_L=0.0010 (6s)
[6/432] T13|optimistic|P_today|ENC_ACTIVE|1        p_L=0.0037 (7s)
[7/432] T13|optimistic|P_archive|BARE|0            p_L=0.0080 (8s)
[8/432] T13|optimistic|P_archive|BARE|1            p_L=0.0467 (9s)
[9/432] T13|optimistic|P_archive|ENC_PASSIVE|0     p_L=0.0057 (11s)
[10/432] T13|optimistic|P_archive|ENC_PASSIVE|1     p_L=0.0112 (12s)
[11/432] T13|optimistic|P_archive|ENC_ACTIVE|0      p_L=0.0030 (13s)
[12/432] T13|optimistic|P_archive|ENC_ACTIVE|1      p_L=0.0053 (14s)
[13/432] T13|optimistic|P_weak|BARE|0               p_L=0.0088 (15s)
[14/432] T13|optimistic|P_weak|BARE|1 